# Módulo 12: Módulos y paquetes

Autor: Juliho Castillo Colmenares

> Lección complementaria del ejercicio [m12-modulos-y-paquetes.ipynb](m12-modulos-y-paquetes.ipynb). No es una tarea entregada del programa.

Hasta ahora todo el código vivía en un solo notebook. En cuanto un proyecto crece, conviene repartirlo en archivos reutilizables y apoyarse en el código que ya escribieron otros. En esta lección vemos cómo se organiza y se importa el código en Python, cómo se localizan los archivos en disco y cómo se lee un CSV con la librería estándar, usando `estatura-por-pais.csv` (estatura media de hombres y mujeres en 199 países).

1. Módulo, paquete y librería
2. Formas de `import`
3. Un recorrido por la librería estándar
4. Directorio de trabajo y rutas
5. Abrir archivos con `with open`
6. Leer un CSV con el módulo `csv`
7. Crear un módulo propio
8. Instalar paquetes con uv
9. Errores frecuentes
10. Ejercicio

## 1. Módulo, paquete y librería

| Término | Qué es | Ejemplo |
| --- | --- | --- |
| **Módulo** | Un archivo `.py` con funciones, clases y variables | `math`, `csv`, o un `utilidades.py` propio |
| **Paquete** | Una carpeta que agrupa módulos (y subpaquetes) | `pandas`, con submódulos como `pandas.io` |
| **Librería** | Término general para un conjunto de código reutilizable; puede ser un módulo o varios paquetes | La **librería estándar** de Python, o NumPy |

La **librería estándar** viene incluida con Python y no requiere instalación. Los paquetes de terceros, como pandas o seaborn, se instalan aparte.

## 2. Formas de `import`

| Forma | Uso posterior | Cuándo conviene |
| --- | --- | --- |
| `import math` | `math.sqrt(2)` | Por defecto: deja claro de dónde viene cada nombre |
| `from math import sqrt, pi` | `sqrt(2)` | Cuando se usan pocos nombres muchas veces |
| `import numpy as np` | `np.array(...)` | Nombres largos con alias convencionales (`np`, `pd`, `plt`) |
| `from math import *` | `sqrt(2)` | Casi nunca: llena el espacio de nombres y oculta el origen |

Un módulo se ejecuta una sola vez, la primera vez que se importa; las importaciones siguientes reutilizan el objeto ya cargado, que queda guardado en `sys.modules`.

In [1]:
import math
from math import pi, sqrt
import statistics as st

print(math.sqrt(16), sqrt(16))
print(round(pi, 4))
print(st.median([3, 1, 4, 1, 5]))

print(type(math).__name__, math.__name__)    # un módulo también es un objeto
print([nombre for nombre in dir(math) if nombre.startswith("log")])

4.0 4.0
3.1416
3
module math
['log', 'log10', 'log1p', 'log2']


## 3. Un recorrido por la librería estándar

Algunos módulos que aparecen una y otra vez en ciencia de datos:

| Módulo | Para qué sirve |
| --- | --- |
| `os` | Interactuar con el sistema operativo: directorio actual, variables de entorno, listar carpetas |
| `pathlib` | Manejar rutas de archivos como objetos (`Path`) |
| `csv` | Leer y escribir archivos CSV |
| `math`, `statistics` | Funciones matemáticas y estadísticas básicas |
| `functools` | Herramientas para funciones: `reduce`, `partial`, `lru_cache` |
| `collections` | Estructuras extra: `Counter`, `defaultdict`, `namedtuple` |
| `datetime` | Fechas y horas |

In [2]:
import os
from collections import Counter
from datetime import date
from functools import partial

print("Separador de rutas en este sistema:", repr(os.sep))
print("Días hasta fin de año desde el 1 de octubre de 2026:", (date(2026, 12, 31) - date(2026, 10, 1)).days)
print(Counter("mississippi").most_common(2))

# partial fija algunos argumentos de una función y devuelve otra
redondear_1 = partial(round, ndigits=1)
print(redondear_1(3.14159))

Separador de rutas en este sistema: '\\'
Días hasta fin de año desde el 1 de octubre de 2026: 91
[('i', 4), ('s', 4)]
3.1


## 4. Directorio de trabajo y rutas

El **directorio de trabajo** es la carpeta desde la que Python resuelve las rutas relativas. En un notebook suele ser la carpeta donde está el `.ipynb`. `os.getcwd()` lo devuelve como texto y `Path.cwd()` como objeto `Path`. Aquí imprimimos solo el nombre de la carpeta para no mostrar la ruta completa del equipo.

| Ruta | Ejemplo | Se interpreta desde |
| --- | --- | --- |
| Relativa | `estatura-por-pais.csv`, `datos/ventas.csv`, `../otra.csv` | El directorio de trabajo |
| Absoluta | `C:/Users/ana/datos/ventas.csv`, `/home/ana/datos/ventas.csv` | La raíz del disco |

In [3]:
from pathlib import Path

print("os.getcwd() termina en:", os.path.basename(os.getcwd()))
print("Path.cwd().name:", Path.cwd().name)

ruta = Path("estatura-por-pais.csv")
print("¿Existe?", ruta.exists())
print("Nombre:", ruta.name, "| raíz:", ruta.stem, "| extensión:", ruta.suffix)

# El operador / une partes de una ruta con el separador correcto del sistema
print(Path("datos") / "2026" / "ventas.csv")

os.getcwd() termina en: m12-modulos-y-paquetes
Path.cwd().name: m12-modulos-y-paquetes
¿Existe? True
Nombre: estatura-por-pais.csv | raíz: estatura-por-pais | extensión: .csv
datos\2026\ventas.csv


Para cambiar el directorio de trabajo existe `os.chdir(ruta)`. Después de llamarlo, **todas** las rutas relativas se interpretan desde la nueva carpeta, también las de celdas que ya funcionaban. Por eso, en lugar de cambiar de directorio, suele ser más seguro construir las rutas a partir de una carpeta base:

```python
os.chdir("C:/Users/ana/proyecto")      # cambia el directorio para todo el proceso
os.chdir("..")                         # sube un nivel

BASE = Path("C:/Users/ana/proyecto")   # alternativa: rutas explícitas
datos = BASE / "datos" / "ventas.csv"
```

En esta lección no cambiamos de directorio, para no romper las rutas relativas de las celdas siguientes.

## 5. Abrir archivos con `with open`

`open(ruta, modo, encoding=...)` abre un archivo y devuelve un objeto con el que se lee o escribe. El bloque `with` garantiza que el archivo se cierre al terminar, aunque ocurra un error en medio.

| Modo | Significado |
| --- | --- |
| `"r"` | Leer (por defecto) |
| `"w"` | Escribir, borrando el contenido anterior |
| `"a"` | Añadir al final |

Conviene indicar siempre el `encoding`. Si se omite, Python usa el del sistema operativo, que en Windows suele ser `cp1252` y en Linux o macOS, `utf-8`: el mismo código puede funcionar en un equipo y fallar en otro.

In [4]:
with open("estatura-por-pais.csv", encoding="utf-8") as archivo:
    encabezado = archivo.readline().strip()
    primeras = [archivo.readline().strip() for _ in range(3)]

print(encabezado)
print(*primeras, sep="\n")
print("¿Cerrado al salir del with?", archivo.closed)

Rank,Country Name,Male Height in Cm,Female Height in Cm,Male Height in Ft,Female Height in Ft
1,Netherlands,183.78,170.36,6.03,5.59
2,Montenegro,183.30,169.96,6.01,5.58
3,Estonia,182.79,168.66,6.00,5.53
¿Cerrado al salir del with? True


## 6. Leer un CSV con el módulo `csv`

Partir cada línea por comas con `split(",")` falla en cuanto un campo contiene una coma entre comillas, como `"Bonaire, San Eustaquio y Saba"`. El módulo `csv` resuelve esos casos:

- `csv.reader` devuelve cada fila como una **lista** de textos.
- `csv.DictReader` usa la primera fila como encabezado y devuelve cada fila como un **diccionario** `{columna: valor}`, de modo que se accede por nombre y no por posición.

Todos los valores llegan como **texto**: hay que convertirlos con `float()` o `int()` antes de operar. Se recomienda abrir el archivo con `newline=""`, como indica la documentación del módulo.

In [5]:
import csv

with open("estatura-por-pais.csv", encoding="utf-8", newline="") as archivo:
    filas = list(csv.DictReader(archivo))

print(len(filas), "filas")
print(filas[0])

diferencias = {}
for fila in filas:
    hombres = float(fila["Male Height in Cm"])
    mujeres = float(fila["Female Height in Cm"])
    diferencias[fila["Country Name"]] = hombres - mujeres

mayor = max(diferencias, key=diferencias.get)
menor = min(diferencias, key=diferencias.get)
print(f"Mayor diferencia entre sexos: {mayor} ({diferencias[mayor]:.2f} cm)")
print(f"Menor diferencia entre sexos: {menor} ({diferencias[menor]:.2f} cm)")
print(f"Diferencia media: {sum(diferencias.values()) / len(diferencias):.2f} cm")

199 filas
{'Rank': '1', 'Country Name': 'Netherlands', 'Male Height in Cm': '183.78', 'Female Height in Cm': '170.36', 'Male Height in Ft': '6.03', 'Female Height in Ft': '5.59'}
Mayor diferencia entre sexos: Puerto Rico (16.42 cm)
Menor diferencia entre sexos: Mauritania (5.48 cm)
Diferencia media: 12.15 cm


Con pandas, la misma lectura es una línea, y las columnas numéricas ya llegan convertidas. El módulo `csv` sigue siendo útil cuando no se quiere depender de pandas o el archivo es demasiado grande para cargarlo entero.

In [6]:
import pandas as pd

estaturas = pd.read_csv("estatura-por-pais.csv")
dif = estaturas["Male Height in Cm"] - estaturas["Female Height in Cm"]
print(estaturas.dtypes)
print(f"Diferencia media con pandas: {dif.mean():.2f} cm")

Rank                     int64
Country Name            object
Male Height in Cm      float64
Female Height in Cm    float64
Male Height in Ft      float64
Female Height in Ft    float64
dtype: object
Diferencia media con pandas: 12.15 cm


## 7. Crear un módulo propio

Cualquier archivo `.py` es un módulo. Para importarlo, su carpeta debe estar en la lista `sys.path`, donde Python busca los módulos (la primera entrada suele ser el directorio del script o del notebook). Para no dejar archivos en el repositorio, aquí escribimos el módulo en una carpeta temporal con `tempfile.TemporaryDirectory`, que se borra al salir del `with`.

Cada módulo tiene una variable `__name__`. Vale `"__main__"` cuando el archivo se ejecuta directamente (`python modulo.py`) y el nombre del módulo cuando se importa. El bloque `if __name__ == "__main__":` permite incluir pruebas o una demostración que **no** se ejecutan al importar.

In [7]:
import subprocess
import sys
import tempfile

codigo = '''
"""Utilidades de conversión de estaturas."""

CM_POR_PULGADA = 2.54

def cm_a_pies_y_pulgadas(cm):
    """Convierte centímetros a (pies, pulgadas)."""
    pulgadas = cm / CM_POR_PULGADA
    return int(pulgadas // 12), round(pulgadas % 12, 1)

print(f"Cargando {__name__}")

if __name__ == "__main__":
    print("Prueba:", cm_a_pies_y_pulgadas(180))
'''

with tempfile.TemporaryDirectory() as carpeta:
    (Path(carpeta) / "estaturas_util.py").write_text(codigo, encoding="utf-8")

    sys.path.insert(0, carpeta)              # hacer visible la carpeta
    try:
        import estaturas_util                 # al importar: no corre el bloque __main__
        print(estaturas_util.cm_a_pies_y_pulgadas(170.29))
        print(estaturas_util.__doc__)

        # Ejecutado como programa: __name__ vale "__main__"
        salida = subprocess.run([sys.executable, "estaturas_util.py"], cwd=carpeta,
                                capture_output=True, text=True)
        print(salida.stdout.strip())
    finally:
        sys.path.remove(carpeta)              # dejar sys.path como estaba
        sys.modules.pop("estaturas_util", None)

Cargando estaturas_util
(5, 7.0)
Utilidades de conversión de estaturas.
Cargando __main__
Prueba: (5, 10.9)


Un **paquete** propio es una carpeta con varios módulos, normalmente con un archivo `__init__.py`. Si la carpeta `analisis/` contiene `limpieza.py`, se importa con `from analisis import limpieza` o `from analisis.limpieza import quitar_nulos`.

## 8. Instalar paquetes con uv

Los paquetes de terceros se descargan del índice público **PyPI**. Este repositorio usa **uv**, un gestor que crea un entorno virtual por proyecto y registra las dependencias en dos archivos:

- `pyproject.toml`: qué paquetes necesita el proyecto y con qué versiones mínimas.
- `uv.lock`: las versiones exactas instaladas, para que cualquiera reproduzca el mismo entorno.

| Comando (en la terminal, no en el notebook) | Efecto |
| --- | --- |
| `uv add requests` | Instala el paquete y lo añade a `pyproject.toml` y `uv.lock` |
| `uv remove requests` | Lo desinstala y lo quita de ambos archivos |
| `uv sync` | Instala exactamente lo que dice `uv.lock` |
| `uv run python script.py` | Ejecuta un comando dentro del entorno del proyecto |

El equivalente clásico es `pip install requests`, que instala el paquete pero no registra la dependencia en el proyecto. Desde Python se puede consultar qué versión está instalada:

In [8]:
from importlib.metadata import version

for paquete in ["numpy", "pandas", "matplotlib"]:
    print(f"{paquete:11} {version(paquete)}")

numpy       2.4.6
pandas      2.3.3
matplotlib  3.11.2


## 9. Errores frecuentes

**Rutas relativas desde el directorio equivocado.** Una ruta relativa depende del directorio de trabajo, no de dónde está el notebook. Si el archivo "desaparece" tras un `os.chdir` o al ejecutar el código desde otra carpeta, aparece un `FileNotFoundError`. Conviene comprobar `Path.cwd()` y `ruta.exists()`.

In [9]:
try:
    open("datos/estatura-por-pais.csv", encoding="utf-8")
except FileNotFoundError as e:
    print("FileNotFoundError:", e.strerror, "->", e.filename)
print("Directorio de trabajo:", Path.cwd().name)

FileNotFoundError: No such file or directory -> datos/estatura-por-pais.csv
Directorio de trabajo: m12-modulos-y-paquetes


**Operar con los textos del CSV sin convertirlos.** `csv` entrega cadenas, y las cadenas se comparan letra por letra: `"99.5"` es "mayor" que `"160.1"` porque `"9"` va después de `"1"`.

In [10]:
alturas_texto = ["160.1", "99.5", "150.9"]
print(max(alturas_texto))                    # comparación de texto
print(max(float(a) for a in alturas_texto))  # comparación numérica

99.5
160.1


**Leer con una codificación distinta de la que se usó al escribir.** Los caracteres que no son ASCII se corrompen sin dar error:

In [11]:
with tempfile.TemporaryDirectory() as carpeta:
    prueba = Path(carpeta) / "paises.txt"
    prueba.write_text("Perú,México,Colombia", encoding="utf-8")
    print(prueba.read_text(encoding="cp1252"))   # codificación equivocada
    print(prueba.read_text(encoding="utf-8"))    # la correcta

PerÃº,MÃ©xico,Colombia
Perú,México,Colombia


**Llamar a un archivo propio como un módulo de la librería estándar.** Un `csv.py` o `math.py` en la carpeta del proyecto tiene prioridad en `sys.path` y "tapa" al módulo original, lo que produce errores como `module 'csv' has no attribute 'DictReader'`. Para saber de dónde se cargó un módulo se consulta su atributo `__file__`:

In [12]:
print(Path(csv.__file__).name, "en", Path(csv.__file__).parent.name)

csv.py en Lib


**Olvidar el bloque `if __name__ == "__main__":`.** Sin él, cualquier prueba o `print` del módulo se ejecuta cada vez que alguien lo importa.

## 10. Ejercicio

Con `estatura-por-pais.csv`:

1. Construye la ruta con `pathlib.Path`, comprueba que existe e imprime el directorio de trabajo (solo el nombre de la carpeta).
2. Lee el archivo con `csv.DictReader` y crea un diccionario `{país: estatura media de las mujeres}` con valores numéricos.
3. Calcula la media, el mínimo y el máximo de ese diccionario, indicando el país del mínimo y del máximo.
4. Para estos países de América Latina, muestra la estatura de las mujeres ordenada de mayor a menor: Argentina, Bolivia, Brazil, Chile, Colombia, Ecuador, Guatemala, Mexico, Peru y Uruguay.
5. Escribe en una carpeta temporal un módulo `conversion.py` con una función `cm_a_metros(cm)` y un bloque `if __name__ == "__main__":`, impórtalo y úsalo para expresar la media del punto 3 en metros.
6. Comprueba la media con pandas.

### Solución

In [13]:
ruta = Path("estatura-por-pais.csv")
print("Directorio:", Path.cwd().name, "| ¿existe el archivo?", ruta.exists())

with open(ruta, encoding="utf-8", newline="") as archivo:
    mujeres = {fila["Country Name"]: float(fila["Female Height in Cm"])
               for fila in csv.DictReader(archivo)}

media = sum(mujeres.values()) / len(mujeres)
pais_min = min(mujeres, key=mujeres.get)
pais_max = max(mujeres, key=mujeres.get)
print(f"Media: {media:.2f} cm")
print(f"Mínimo: {pais_min} ({mujeres[pais_min]} cm); máximo: {pais_max} ({mujeres[pais_max]} cm)")

latam = ["Argentina", "Bolivia", "Brazil", "Chile", "Colombia",
         "Ecuador", "Guatemala", "Mexico", "Peru", "Uruguay"]
for pais in sorted(latam, key=mujeres.get, reverse=True):
    print(f"  {pais:10} {mujeres[pais]:.2f} cm")

Directorio: m12-modulos-y-paquetes | ¿existe el archivo? True
Media: 160.94 cm
Mínimo: Guatemala (150.91 cm); máximo: Netherlands (170.36 cm)
  Brazil     162.41 cm
  Uruguay    161.56 cm
  Argentina  161.22 cm
  Chile      159.42 cm
  Colombia   157.96 cm
  Mexico     157.90 cm
  Bolivia    155.58 cm
  Ecuador    155.25 cm
  Peru       154.39 cm
  Guatemala  150.91 cm


In [14]:
codigo = '''
def cm_a_metros(cm):
    return cm / 100

if __name__ == "__main__":
    print(cm_a_metros(175))
'''

with tempfile.TemporaryDirectory() as carpeta:
    (Path(carpeta) / "conversion.py").write_text(codigo, encoding="utf-8")
    sys.path.insert(0, carpeta)
    try:
        from conversion import cm_a_metros
        print(f"Media en metros: {cm_a_metros(media):.3f} m")
    finally:
        sys.path.remove(carpeta)
        sys.modules.pop("conversion", None)

print(f"Media con pandas: {pd.read_csv(ruta)['Female Height in Cm'].mean():.2f} cm")

Media en metros: 1.609 m
Media con pandas: 160.94 cm


## Resumen

- Un **módulo** es un archivo `.py`; un **paquete**, una carpeta de módulos; la **librería estándar** viene con Python.
- `import x`, `from x import y` e `import x as alias` cambian cómo se escribe el nombre, no qué se carga; `from x import *` conviene evitarlo.
- Las rutas relativas dependen del **directorio de trabajo**; `pathlib.Path` facilita construirlas y comprobarlas, y `os.chdir` cambia ese directorio para todo el proceso.
- `with open(..., encoding=...)` cierra el archivo solo; `csv.DictReader` da filas como diccionarios de **texto** que hay que convertir.
- Un módulo propio se importa si su carpeta está en `sys.path`; `if __name__ == "__main__":` separa el código de prueba.
- Los paquetes de terceros se instalan con `uv add`, que registra la dependencia en `pyproject.toml` y `uv.lock`.

En el **Módulo 13** comienza la sección de ciencia de datos con su historia, su contexto y sus definiciones básicas.